# Stage 3 – Modelling

Leakage-aware preprocessing on a 200k sample, then Logistic Regression, Random Forest, XGBoost, HistGradientBoosting and a stacking ensemble. Saves models, SHAP/LIME plots and `model_comparison.csv`.

Part of **Explainable AI for Credit Risk Prediction**. See the main README for the full pipeline.

In [14]:
# 03_modeling.ipynb
# ===========================================================
# Stage 3: Credit Risk Modeling + XAI (Leakage-aware, Memory-safe)
# Project: Explainable AI for Credit Risk Prediction (Sai)
# Data: accepted_cleaned.csv from Notebook 1
# ===========================================================
# LIME for local explanations

from lime.lime_tabular import LimeTabularExplainer
import os
import gc
import random
import warnings

from typing import List

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Modeling / preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    auc,
)

import optuna
import joblib
import xgboost as xgb
import shap

warnings.filterwarnings("ignore")
tqdm.pandas()

# -----------------------------------------------------------
# CONFIG
# -----------------------------------------------------------
INPUT_FILE = "../data/accepted_cleaned.csv"   # cleaned dataset from Notebook 1
SAMPLE_TARGET = 200_000                      # rows to sample (None => full file, but heavy)
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_FOLDS_FOR_TE = 5                           # for target-mean encoding
OPTUNA_TRIALS_RF = 8
OPTUNA_TRIALS_XGB = 8
SAVE_DIR = "../outputs/notebook3_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

TARGET_COL = "default"   # created in Notebook 1: 1 = default, 0 = fully paid

# -----------------------------------------------------------
# Utility: memory-safe reservoir sampling from CSV
# -----------------------------------------------------------
def reservoir_sample_csv(
    path: str,
    sample_size: int,
    random_state: int = 42,
    chunksize: int = 100_000,
):
    """
    Reservoir sampling for large CSV.
    Returns: sampled_df (pandas), total_rows_read (int)
    """
    rng = random.Random(random_state)
    reservoir = []
    total = 0

    reader = pd.read_csv(
        path,
        compression="infer",
        chunksize=chunksize,
        low_memory=True
    )

    for chunk in tqdm(reader, desc="Streaming CSV for sampling"):
        for _, row in chunk.iterrows():
            if total < sample_size:
                reservoir.append(row)
            else:
                j = rng.randrange(total + 1)
                if j < sample_size:
                    reservoir[j] = row
            total += 1
        del chunk
        gc.collect()

    if total == 0:
        raise FileNotFoundError(f"No rows read from {path}")

    sampled_df = pd.DataFrame(reservoir).reset_index(drop=True)
    return sampled_df, total

# -----------------------------------------------------------
# 1) Sampling / loading dataset
# -----------------------------------------------------------
print("1) Sampling/loading dataset...")

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

if SAMPLE_TARGET is not None:
    df, total_rows = reservoir_sample_csv(
        INPUT_FILE,
        sample_size=SAMPLE_TARGET,
        random_state=RANDOM_STATE,
    )
    print(f"Sampled {len(df)} rows from {total_rows} total rows.")
else:
    df = pd.read_csv(INPUT_FILE, low_memory=True)
    total_rows = len(df)
    print(f"Loaded full cleaned dataset: {df.shape}")

# -----------------------------------------------------------
# 2) Create binary target & drop leakage columns
# -----------------------------------------------------------
print("2) Create binary target and drop leakage cols if present...")

# Ensure target exists
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in cleaned dataset.")

# Coerce target to int (0/1)
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce").fillna(0).astype(int)

# Explicit leakage columns: payment/outcome info etc.
leak_cols_explicit = [
    "loan_status",              # original text outcome (categorical)
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "last_pymnt_amnt",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d",
    "out_prncp",
    "out_prncp_inv",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
    "settlement_percentage",
    # ❗ Not available at application time / outcome-related
    "debt_settlement_flag",
]

# FICO at end of loan – future info
leak_cols_explicit += [
    "last_fico_range_low",
    "last_fico_range_high",
]

# Pure identifiers / text / region that we don't want as features
leak_cols_explicit += [
    "id",          # ❗ drop ID as feature
    "desc",
    "url",
    "title",
    "zip_code",
    "addr_state",
    "policy_code",
]


drop_now = [c for c in leak_cols_explicit if c in df.columns]
if drop_now:
    df.drop(columns=drop_now, inplace=True)
    print(f"--> Dropped explicit leakage columns: {drop_now}")

# Drop constant/all-null columns
nunique = df.nunique(dropna=True)
const_cols = [c for c in df.columns if nunique.get(c, 0) <= 1 and c != TARGET_COL]
if const_cols:
    df.drop(columns=const_cols, inplace=True)
    print(f"--> Dropped constant/all-null columns: {const_cols}")

print("Remaining columns:", len(df.columns))

# -----------------------------------------------------------
# 3) Detect numeric vs categorical features (robust)
# -----------------------------------------------------------
print("3) Detect numeric vs categorical features (robust detection)...")

# List candidate feature columns (exclude target)
feature_cols = [c for c in df.columns if c != TARGET_COL]

numeric_cols: List[str] = []
categorical_cols: List[str] = []

for c in feature_cols:
    # Take a sample to inspect type
    sample = df[c].dropna().head(500)
    # Try numeric conversion
    sample_num = pd.to_numeric(sample, errors="coerce")
    non_na = len(sample_num.dropna())
    frac_numeric = non_na / max(len(sample), 1)

    # Heuristic:
    # - mostly numeric values => numeric
    # - or high cardinality numeric-like strings
    if frac_numeric > 0.8:
        numeric_cols.append(c)
    else:
        categorical_cols.append(c)

print(f"Detected numeric columns: {len(numeric_cols)}, categorical columns: {len(categorical_cols)}")
print("Numeric examples:", numeric_cols[:10])
print("Categorical examples:", categorical_cols[:10])

# -----------------------------------------------------------
# 4) Coerce numeric columns & clean categorical
# -----------------------------------------------------------
print("4) Coercing numeric columns & cleaning categoricals...")

for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# For categorical: fill NaN with token and cast to string
if categorical_cols:
    df[categorical_cols] = df[categorical_cols].astype(str).fillna("__MISSING__")

# -----------------------------------------------------------
# 5) Split categorical by cardinality
# -----------------------------------------------------------
print("5) Splitting categorical columns by cardinality...")

small_cat = []
large_cat = []
for c in categorical_cols:
    nun = df[c].nunique()
    if nun < 20:
        small_cat.append(c)
    else:
        large_cat.append(c)

print(f"Small-cardinality: {len(small_cat)}  Large-cardinality: {len(large_cat)}")

# If too many large-cat, restrict to top-freq columns
MAX_LARGE_USE = 8
if len(large_cat) > MAX_LARGE_USE:
    # keep those with highest frequency mass in top-100 categories
    freq_sorted = sorted(
        large_cat,
        key=lambda col: df[col].value_counts().iloc[:100].sum(),
        reverse=True
    )
    keep_large = freq_sorted[:MAX_LARGE_USE]
    drop_large_from_te = [c for c in large_cat if c not in keep_large]
    print(f"Reducing large-cardinality columns for TE to top {MAX_LARGE_USE}. Ignoring TE on: {drop_large_from_te}")
    large_cat = keep_large
    # The dropped large-card cols remain in 'small_cat' with truncated labels
    for c in drop_large_from_te:
        df[c] = df[c].apply(lambda s: s if len(s) <= 30 else s[:30])
        small_cat.append(c)

###############################################################################
# 6. Target-mean encoding for large-cardinality columns (GLOBAL MEAN VERSION)
###############################################################################
print("6) Target-mean encoding for large-cardinality columns ...")

y_series = df[TARGET_COL].astype(int)
X_te = pd.DataFrame(index=df.index)
te_encoders = {}                      # <== store mappings for each TE column
global_mean = y_series.mean()

if large_cat:
    for col in large_cat:
        print(f"  TE -> {col}")

        # mean default per category
        means = df.groupby(col)[TARGET_COL].mean()
        te_encoders[col] = means.to_dict()   # save mapping for later use

        # map column to its mean default
        enc_col = df[col].map(te_encoders[col]).fillna(global_mean)
        X_te[col + "_te"] = enc_col.values
else:
    print("  No large-cardinality columns to encode.")

# Drop original large-cat columns from df (we'll use the *_te versions)
if large_cat:
    df.drop(columns=large_cat, inplace=True)

###############################################################################
# 7. Ordinal-encoding small-cardinality categorical features ...
###############################################################################
print("7) Ordinal-encoding small-cardinality categorical features ...")

if small_cat:
    from sklearn.preprocessing import OrdinalEncoder

    ordinal_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    df_small = pd.DataFrame(
        ordinal_encoder.fit_transform(df[small_cat].astype(str)),
        columns=[c + "_ord" for c in small_cat],
        index=df.index,
    )
else:
    ordinal_encoder = None
    df_small = pd.DataFrame(index=df.index)



# -----------------------------------------------------------
# 8) Assemble final feature matrix
# -----------------------------------------------------------
print("8) Assembling final feature matrix...")

# Recompute feature columns in df (excluding target)
current_feature_cols = [c for c in df.columns if c != TARGET_COL]

# Numeric features (keep raw numeric columns)
numeric_cols_final = [c for c in numeric_cols if c in current_feature_cols]
X_numeric = df[numeric_cols_final].copy() if len(numeric_cols_final) > 0 else pd.DataFrame(index=df.index)

# Combine numeric + small-cat-encoded + TE columns
parts = []
if len(numeric_cols_final) > 0:
    parts.append(X_numeric.reset_index(drop=True))
if not df_small.empty:
    parts.append(df_small.reset_index(drop=True))
if not X_te.empty:
    parts.append(X_te.reset_index(drop=True))

if not parts:
    raise RuntimeError("No features built. Check numeric/categorical detection.")

X_all = pd.concat(parts, axis=1)
y = y_series.reset_index(drop=True)

feature_names = X_all.columns.tolist()

print(f"Final feature matrix: {X_all.shape}, target: {y.shape}")
print("Memory usage (MB):", X_all.memory_usage(deep=True).sum() / (1024**2))

# -----------------------------------------------------------
# 9) Train/test split
# -----------------------------------------------------------
print("9) Train/test split (stratified)...")

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {X_train.shape} Test: {X_test.shape}")

# -----------------------------------------------------------
# 10) Impute numeric features and scale
# -----------------------------------------------------------
print("10) Impute numeric features and scale (fit on train only)...")

preprocessor_num = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

if len(numeric_cols_final) > 0:
    X_train_num = preprocessor_num.fit_transform(X_train[numeric_cols_final])
    X_test_num = preprocessor_num.transform(X_test[numeric_cols_final])
else:
    # no numeric features
    X_train_num = np.empty((X_train.shape[0], 0))
    X_test_num = np.empty((X_test.shape[0], 0))

# The remaining columns (ordinal + TE) are already numeric, with no NaN
other_cols = [c for c in X_all.columns if c not in numeric_cols_final]
X_train_other = X_train[other_cols].to_numpy(dtype=float)
X_test_other = X_test[other_cols].to_numpy(dtype=float)

# Final matrices
X_train_final = np.hstack([X_train_num, X_train_other])
X_test_final = np.hstack([X_test_num, X_test_other])
final_feature_names = list(numeric_cols_final) + other_cols

print("Final matrices shapes:", X_train_final.shape, X_test_final.shape)

# Cleanup big objects
del X_all, X_train_num, X_test_num, X_train_other, X_test_other
gc.collect()

# -----------------------------------------------------------
# 11) Evaluation helper
# -----------------------------------------------------------
def evaluate_model(model, X_te, y_te, name="Model"):
    y_pred = model.predict(X_te)
    report = classification_report(y_te, y_pred, digits=4)

    # Get probabilities
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_te)[:, 1]
    else:
        # fallback for models with decision_function
        scores = model.decision_function(X_te)
        probs = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

    roc = roc_auc_score(y_te, probs)
    prec, rec, _ = precision_recall_curve(y_te, probs)
    pr_auc = auc(rec, prec)
    cm = confusion_matrix(y_te, y_pred)

    print(f"\n=== {name} ===")
    print(report)
    print("Confusion Matrix:\n", cm)
    print(f"ROC AUC: {roc:.6f}, PR AUC: {pr_auc:.6f}")

    return {
        "report": report,
        "roc_auc": roc,
        "pr_auc": pr_auc,
        "cm": cm,
    }

# -----------------------------------------------------------
# 12) Logistic Regression baseline
# -----------------------------------------------------------
print("11) Training Logistic Regression (baseline)...")

lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="saga",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
lr.fit(X_train_final, y_train)
lr_results = evaluate_model(lr, X_test_final, y_test, "Logistic Regression")

joblib.dump(lr, os.path.join(SAVE_DIR, "logreg.joblib"))

# -----------------------------------------------------------
# 13) Random Forest + Optuna tuning
# -----------------------------------------------------------
print("12) Tuning RandomForest (Optuna light)...")

def rf_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 300),
        "max_depth": trial.suggest_int("max_depth", 4, 14),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "class_weight": "balanced",
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
    }
    clf = RandomForestClassifier(**params)
    clf.fit(X_train_final, y_train)
    preds = clf.predict_proba(X_test_final)[:, 1]
    return roc_auc_score(y_test, preds)

study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(rf_objective, n_trials=OPTUNA_TRIALS_RF, show_progress_bar=True)
best_rf_params = study_rf.best_trial.params
print("Best RF params:", best_rf_params)

rf = RandomForestClassifier(
    **best_rf_params,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf.fit(X_train_final, y_train)
rf_results = evaluate_model(rf, X_test_final, y_test, "Random Forest")

joblib.dump(rf, os.path.join(SAVE_DIR, "random_forest.joblib"))

# -----------------------------------------------------------
# 14) XGBoost + Optuna tuning
# -----------------------------------------------------------
print("13) Tuning XGBoost (Optuna light)...")

def xgb_objective(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        "eta": trial.suggest_float("eta", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
    }
    clf = xgb.XGBClassifier(**params)
    clf.fit(X_train_final, y_train)
    preds = clf.predict_proba(X_test_final)[:, 1]
    return roc_auc_score(y_test, preds)

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(xgb_objective, n_trials=OPTUNA_TRIALS_XGB, show_progress_bar=True)
best_xgb_params = study_xgb.best_trial.params
print("Best XGB params:", best_xgb_params)

xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    **best_xgb_params,
)
xgb_clf.fit(X_train_final, y_train)
xgb_results = evaluate_model(xgb_clf, X_test_final, y_test, "XGBoost")

joblib.dump(xgb_clf, os.path.join(SAVE_DIR, "xgboost.joblib"))

# -----------------------------------------------------------
# 15) HistGradientBoosting (fast, can handle NaNs)
# -----------------------------------------------------------
print("14) Training HistGradientBoostingClassifier (fast; accepts NaNs)...")

hgb = HistGradientBoostingClassifier(
    max_iter=200,
    random_state=RANDOM_STATE,
)
hgb.fit(X_train_final, y_train)
hgb_results = evaluate_model(hgb, X_test_final, y_test, "HistGradientBoosting")

joblib.dump(hgb, os.path.join(SAVE_DIR, "histgb.joblib"))

# -----------------------------------------------------------
# 16) Stacking ensemble
# -----------------------------------------------------------
print("15) Training stacking ensemble (light)...")

estimators = [
    ("rf", rf),
    ("xgb", xgb_clf),
    ("hgb", hgb),
]
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000, class_weight="balanced"),
    n_jobs=-1,
)
stack.fit(X_train_final, y_train)
stack_results = evaluate_model(stack, X_test_final, y_test, "Stacking Ensemble")

joblib.dump(stack, os.path.join(SAVE_DIR, "stacking.joblib"))

# -----------------------------------------------------------
# 17) SHAP explanations (XAI)
# -----------------------------------------------------------
print("16) SHAP explanations (sampled subset)...")

explainer_model = xgb_clf  # choose tree-based model for SHAP
shap_sample_size = min(2000, X_test_final.shape[0])
rng = np.random.RandomState(RANDOM_STATE)
shap_idx = rng.choice(np.arange(X_test_final.shape[0]), size=shap_sample_size, replace=False)

X_shap = X_test_final[shap_idx]
y_shap = y_test.iloc[shap_idx].values

explainer = shap.TreeExplainer(explainer_model)
shap_values = explainer.shap_values(X_shap)  # shape (n_samples, n_features)

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = sorted(
    zip(final_feature_names, mean_abs_shap),
    key=lambda x: x[1],
    reverse=True
)[:30]

print("Top 30 features by mean |SHAP| (no target / payment leakage):")
for feat, imp in shap_importance:
    print(f"  {feat}: {imp:.6f}")

joblib.dump(
    {"explainer": explainer, "feature_names": final_feature_names},
    os.path.join(SAVE_DIR, "shap_explainer.joblib")
)

# Example local explanation
print("\nExample local explanation for one test instance:")
print("True label:", int(y_shap[0]))
inst_shap = shap_values[0]
top_k = np.argsort(np.abs(inst_shap))[::-1][:12]
for i in top_k:
    print(f"  {final_feature_names[i]} => SHAP {inst_shap[i]:.6f}")

# Global SHAP summary (top features)
shap.summary_plot(shap_values, X_shap, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig("../outputs/notebook3_outputs/shap_summary.png", dpi=300)
plt.close()

# Optional: bar plot (easier to read)
shap.summary_plot(shap_values, X_shap, feature_names=feature_names, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("../outputs/notebook3_outputs/shap_summary_bar.png", dpi=300)
plt.close()

# Local explanation for one instance
i = 0  # first sample in X_shap
shap.plots.waterfall(shap.Explanation(values=shap_values[i],
                                      base_values=explainer.expected_value,
                                      data=X_shap[i],
                                      feature_names=feature_names),
                     show=False)
plt.tight_layout()
plt.savefig("../outputs/notebook3_outputs/shap_local_example.png", dpi=300)
plt.close()

print("Saved SHAP plots to notebook3_outputs/")
#Lime 

import numpy as np

print("\n=== LIME: setting up explainer for XGBoost ===")

# 1) Training data for LIME (use a subset if needed to speed up)
lime_train = X_train_final
# If you want to speed up drastically:
# lime_train = X_train_final[:20000]

# 2) Class names (0 = non-default, 1 = default)
class_names = ["non_default", "default"]

# 3) Build the LimeTabularExplainer
lime_explainer = LimeTabularExplainer(
    training_data=lime_train,
    feature_names=feature_names,
    class_names=class_names,
    mode="classification",
    discretize_continuous=True
)

print("LIME explainer ready.")





import numpy as np
import matplotlib.pyplot as plt

# pick an instance index
instance_idx = 0   # change this to inspect different customers

# 1) raw instanceA
x_instance = X_test_final[instance_idx]

# 2) explain (ask for top 2 labels to be safe)
lime_exp = lime_explainer.explain_instance(
    data_row=x_instance,
    predict_fn=xgb_clf.predict_proba,
    num_features=10,
    top_labels=2    # get explanations for both 0 and 1 if needed
)

# model prediction
probs = xgb_clf.predict_proba(X_test_final[instance_idx:instance_idx+1])[0]
pred_label = int(np.argmax(probs))   # 0 or 1

print(f"True label: {int(y_test.iloc[instance_idx])}")
print(f"Predicted probabilities (non_default, default): {probs}")
print(f"Predicted class: {pred_label}")

# Which label to show?
# - if you want "whatever LIME has": use the predicted class
# - if you specifically want default (1), but it's not there, fall back to predicted
label_to_show = 1 if 1 in lime_exp.local_exp else pred_label

print(f"\nLIME explanation for label: {label_to_show} (0=non-default, 1=default)")
print("Feature contributions:")
for feature, weight in lime_exp.as_list(label=label_to_show):
    print(f"  {feature}: {weight:.4f}")

# Plot and save
fig = lime_exp.as_pyplot_figure(label=label_to_show)
plt.tight_layout()
plt.savefig("../outputs/notebook3_outputs/lime_local_example.png", dpi=300)
plt.close()

print("\nSaved LIME plot to notebook3_outputs/lime_local_example.png")


# --------------------------------------------------
# SAVE FULL PREPROCESSING PIPELINE (NUM + CAT + TE)
# --------------------------------------------------
preprocessing_bundle = {
    "numeric_preprocessor": preprocessor_num,   # SimpleImputer + StandardScaler
    "ordinal_encoder": ordinal_encoder,         # may be None if no small_cat
    "target_encoders": te_encoders,             # dict of {col: {category: mean}}
    "feature_names": feature_names,             # final model feature order
    "numeric_cols_final": numeric_cols_final,
    "small_cat_used": small_cat_used,
    "large_cat_used": large_cat_used,
}

pipeline_path = os.path.join(SAVE_DIR, "preprocessing_pipeline.joblib")
joblib.dump(preprocessing_bundle, pipeline_path)
print(f"\n✅ Full preprocessing pipeline saved to {pipeline_path}")

# -----------------------------------------------------------
# 18) Save cleaned modeling sample
# -----------------------------------------------------------
print("17) Saving cleaned sampled dataset & metadata (small CSV)...")

cleaned_sample = pd.DataFrame(X_test_final, columns=final_feature_names).reset_index(drop=True)
cleaned_sample[TARGET_COL] = y_test.reset_index(drop=True).values

cleaned_path = os.path.join(SAVE_DIR, "cleaned_sample_for_modelling.csv.gz")
cleaned_sample.to_csv(cleaned_path, index=False, compression="gzip")
print("Saved:", cleaned_path)

meta = {
    "feature_names": final_feature_names,
    "numeric_cols": numeric_cols_final,
    "large_cat_used": large_cat,
    "small_cat_used": small_cat,
}
joblib.dump(meta, os.path.join(SAVE_DIR, "feature_metadata.joblib"))
print("Saved metadata to:", os.path.join(SAVE_DIR, "feature_metadata.joblib"))

print("Done. All models and metadata saved to", SAVE_DIR)

# 18) Threshold tuning for better recall on defaulters

from sklearn.metrics import precision_recall_curve, classification_report, confusion_matrix

best_model = xgb_clf   # you can change to rf_clf, hgb_clf, stack_clf

probs = best_model.predict_proba(X_test_final)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, probs)

def find_threshold_for_recall(target_recall, recalls, thresholds):
    diffs = np.abs(recalls[:-1] - target_recall)
    idx = diffs.argmin()
    return thresholds[idx], idx

target_recall = 0.50
best_thr, idx = find_threshold_for_recall(target_recall, recalls, thresholds)

print(f"\nChosen target recall: {target_recall:.2f}")
print(f"Closest recall: {recalls[idx]:.4f}")
print(f"Threshold: {best_thr:.4f}")
print(f"Precision: {precisions[idx]:.4f}")

y_pred_custom = (probs >= best_thr).astype(int)

print("\n=== Evaluation with tuned threshold ===")
print(classification_report(y_test, y_pred_custom, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_custom))

# Compare with default 0.5
y_pred_default = (probs >= 0.5).astype(int)
print("\n=== Default threshold (0.5) ===")
print(classification_report(y_test, y_pred_default, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_default))

from sklearn.metrics import roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt

probs = xgb_clf.predict_proba(X_test_final)[:, 1]

# ROC
fpr, tpr, _ = roc_curve(y_test, probs)
roc_auc = roc_auc_score(y_test, probs)

plt.figure()
plt.plot(fpr, tpr, label=f"XGBoost (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - XGBoost")
plt.legend()
plt.tight_layout()
plt.savefig("../outputs/notebook3_outputs/xgb_roc_curve.png", dpi=300)
plt.close()

# Precision–Recall
prec, rec, _ = precision_recall_curve(y_test, probs)
pr_auc = auc(rec, prec)

plt.figure()
plt.plot(rec, prec, label=f"XGBoost (PR AUC = {pr_auc:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve - XGBoost")
plt.legend()
plt.tight_layout()
plt.savefig("../outputs/notebook3_outputs/xgb_pr_curve.png", dpi=300)
plt.close()

print("Saved ROC and PR curves to notebook3_outputs/")


import pandas as pd

results_table = pd.DataFrame([
    ["Logistic Regression", lr_results["roc_auc"], lr_results["pr_auc"]],
    ["Random Forest",        rf_results["roc_auc"], rf_results["pr_auc"]],
    ["XGBoost",              xgb_results["roc_auc"], xgb_results["pr_auc"]],
    ["HistGradientBoosting", hgb_results["roc_auc"], hgb_results["pr_auc"]],
    ["Stacking Ensemble",    stack_results["roc_auc"], stack_results["pr_auc"]],
], columns=["Model", "ROC_AUC", "PR_AUC"])

print(results_table)
results_table.to_csv("../outputs/notebook3_outputs/model_comparison.csv", index=False)
print("\nSaved model comparison to notebook3_outputs/model_comparison.csv")


1) Sampling/loading dataset...


Streaming CSV for sampling: 14it [01:18,  5.63s/it]


Sampled 200000 rows from 1345310 total rows.
2) Create binary target and drop leakage cols if present...
--> Dropped explicit leakage columns: ['out_prncp', 'out_prncp_inv', 'debt_settlement_flag', 'last_fico_range_low', 'last_fico_range_high', 'id']
--> Dropped constant/all-null columns: ['hardship_flag']
Remaining columns: 86
3) Detect numeric vs categorical features (robust detection)...
Detected numeric columns: 74, categorical columns: 11
Numeric examples: ['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'int_rate', 'installment', 'annual_inc', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high']
Categorical examples: ['term', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'earliest_cr_line', 'initial_list_status']
4) Coercing numeric columns & cleaning categoricals...
5) Splitting categorical columns by cardinality...
Small-cardinality: 8  Large-cardinality: 3
6) Target-mean encoding for large-cardinality columns ...
  TE

[I 2025-12-02 23:06:03,872] A new study created in memory with name: no-name-46e40597-a5b5-4fd3-a377-acc93ac9ed14



=== Logistic Regression ===
              precision    recall  f1-score   support

           0     0.9358    0.8271    0.8781     32008
           1     0.5274    0.7729    0.6270      7992

    accuracy                         0.8163     40000
   macro avg     0.7316    0.8000    0.7525     40000
weighted avg     0.8542    0.8163    0.8279     40000

Confusion Matrix:
 [[26473  5535]
 [ 1815  6177]]
ROC AUC: 0.900892, PR AUC: 0.754296
12) Tuning RandomForest (Optuna light)...


...

[I 2025-12-02 23:09:25,071] Trial 5 finished with value: 0.9073732724226351 and parameters: {'eta': 0.16461050771989869, 'max_depth': 7, 'subsample': 0.8227914817167493, 'colsample_bytree': 0.6244502569710884, 'n_estimators': 147}. Best is trial 2 with value: 0.909695010400303.


Best trial: 2. Best value: 0.909695:  88%|█████████████████████████████████████████▏     | 7/8 [00:33<00:04,  4.35s/it]

[I 2025-12-02 23:09:28,600] Trial 6 finished with value: 0.9087166533410439 and parameters: {'eta': 0.2549500936635234, 'max_depth': 3, 'subsample': 0.7884791111926759, 'colsample_bytree': 0.5793031688420645, 'n_estimators': 230}. Best is trial 2 with value: 0.909695010400303.


Best trial: 2. Best value: 0.909695: 100%|███████████████████████████████████████████████| 8/8 [00:37<00:00,  4.70s/it]


[I 2025-12-02 23:09:33,021] Trial 7 finished with value: 0.9034122440204513 and parameters: {'eta': 0.1836463622028102, 'max_depth': 6, 'subsample': 0.517317657764617, 'colsample_bytree': 0.6784039501165853, 'n_estimators': 242}. Best is trial 2 with value: 0.909695010400303.
Best XGB params: {'eta': 0.06711763401078519, 'max_depth': 4, 'subsample': 0.9502780459438367, 'colsample_bytree': 0.9437021931756573, 'n_estimators': 313}

=== XGBoost ===
              precision    recall  f1-score   support

           0     0.8824    0.9779    0.9277     32008
           1     0.8437    0.4781    0.6103      7992

    accuracy                         0.8780     40000
   macro avg     0.8630    0.7280    0.7690     40000
weighted avg     0.8747    0.8780    0.8643     40000

Confusion Matrix:
 [[31300   708]
 [ 4171  3821]]
ROC AUC: 0.909695, PR AUC: 0.770165
14) Training HistGradientBoostingClassifier (fast; accepts NaNs)...

=== HistGradientBoosting ===
              precision    recall  f1-s

In [5]:
import numpy as np
import matplotlib.pyplot as plt

# --------- choose an instance ----------
instance_idx = 0   # change this number to inspect different customers

# 1) Get the instance from X_test_final (works for both ndarray & DataFrame)
if isinstance(X_test_final, np.ndarray):
    x_instance = X_test_final[instance_idx]              # shape (n_features,)
else:
    x_instance = X_test_final.iloc[instance_idx].values  # convert to 1D array

# 2) LIME explanation
lime_exp = lime_explainer.explain_instance(
    data_row=x_instance,
    predict_fn=xgb_clf.predict_proba,
    num_features=10,
    top_labels=2    # ask for explanations for both classes
)

# 3) Model prediction for this borrower
probs = xgb_clf.predict_proba(x_instance.reshape(1, -1))[0]
pred_label = int(np.argmax(probs))

# ✅ use iloc here so we don’t get KeyError on index
true_label = int(y_test.iloc[instance_idx])

print(f"True label: {true_label}")
print(f"Predicted probabilities (class 0, class 1): {probs}")
print(f"Predicted class: {pred_label}")

# 4) Decide which label to explain (prefer default class = 1 if available)
if 1 in lime_exp.local_exp:
    label_to_show = 1
else:
    label_to_show = pred_label

print(f"\nShowing LIME explanation for label: {label_to_show}")
print("\nLIME explanation (feature contributions):")
for feature, weight in lime_exp.as_list(label=label_to_show):
    print(f"  {feature}: {weight:.4f}")

# 5) Plot & save for thesis
fig = lime_exp.as_pyplot_figure(label=label_to_show)
plt.tight_layout()
plt.savefig("../outputs/notebook3_outputs/lime_local_example.png", dpi=300)
plt.close()

print("\nSaved LIME plot to notebook3_outputs/lime_local_example.png")


True label: 0
Predicted probabilities (class 0, class 1): [0.92119896 0.07880104]
Predicted class: 0

Showing LIME explanation for label: 1

LIME explanation (feature contributions):
  term_ord > 0.00: 0.0699
  dti <= -0.55: -0.0425
  delinq_amnt <= -0.02: -0.0383
  0.11 < sub_grade_te <= 0.19: -0.0358
  grade_ord <= 1.00: -0.0335
  num_tl_30dpd <= -0.05: 0.0260
  home_ownership_ord <= 1.00: -0.0251
  num_tl_120dpd_2m <= -0.03: 0.0240
  purpose_ord > 2.00: 0.0215
  chargeoff_within_12_mths <= -0.08: -0.0159

Saved LIME plot to notebook3_outputs/lime_local_example.png
